## EDA on label from User

QUESTION 1: Find all possible unique values
- Category
- Subcategory
- Root_code
- Sub_code

In [ ]:
import pandas as pd

In [ ]:

df = pd.read_excel('label.xlsx')

print("--- 1. UNIQUE VALUES IN DATASET ---")
print(f"Categories ({df['Category'].nunique()}): {df['Category'].dropna().unique()}")
print(f"Subcategories ({df['Subcategory'].nunique()}): {df['Subcategory'].dropna().unique()}")
print(f"Root Codes ({df['root_code'].nunique()}): {df['root_code'].dropna().unique()}")
print(f"Sub Codes ({df['sub_code'].nunique()}): {df['sub_code'].dropna().unique()}\n")

QUESTION 2: Find mappings and human errors

In [ ]:
print("--- 2. MAPPING RULES & ANOMALY DETECTION ---")

# Group by the human text labels and see what codes they mapped to, counting occurrences
mapping_df = df.groupby(
    ['Category', 'Subcategory', 'root_code', 'sub_code'], 
    dropna=False
).size().reset_index(name='Count')

# Sort so that for each Category/Subcategory, the most frequent mapping is at the top
mapping_df = mapping_df.sort_values(
    by=['Category', 'Subcategory', 'Count'], 
    ascending=[True, True, False]
)

# Calculate the total times a Category/Subcategory pair appears
mapping_df['Total_Category_Occurrences'] = mapping_df.groupby(['Category', 'Subcategory'])['Count'].transform('sum')

# Calculate the percentage of times THIS specific mapping was used
mapping_df['Mapping_Confidence_%'] = (mapping_df['Count'] / mapping_df['Total_Category_Occurrences']) * 100

print("\nAll Mappings (Sorted by frequency):")
display(mapping_df) # Use print(mapping_df) if not in Jupyter

In [ ]:
# --- ISOLATING THE ERRORS ---
# Assuming that whatever mapping happens the majority of the time (>50%) is the "Correct" rule...
# Anything else is likely a human annotation error.

print("\n--- LIKELY HUMAN ANNOTATION ERRORS ---")
# Filter for mappings that happen less than 20% of the time for that specific category
anomalies = mapping_df[mapping_df['Mapping_Confidence_%'] < 20.0]

if anomalies.empty:
    print("Good news! No major anomalies found. The mapping is consistent.")
else:
    print("Found potential mislabels! Annotators likely messed these up:")
    display(anomalies)

In [ ]:
# ==========================================
# CELL 1: EYEBALL SPECIFIC CATEGORIES
# ==========================================
from IPython.display import Image, display
import os

# 1. Define the specific pair you want to investigate
TARGET_CATEGORY = "Insert Category Here"       # e.g., "Medical Document"
TARGET_SUBCAT = "Insert Subcategory Here"      # e.g., "Lab Results"

# 2. Filter the existing 'df' for this specific pair
subset_df = df[(df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)]

print(f"Found {len(subset_df)} documents for {TARGET_CATEGORY} -> {TARGET_SUBCAT}\n")

# 3. Display the images (Showing the first 10 to avoid crashing the notebook)
for idx, row in subset_df.head(10).iterrows():
    print("-" * 50)
    print(f"Row Index: {idx}")
    print(f"Current Labels -> root_code: [{row['root_code']}] | sub_code: [{row['sub_code']}]")
    print(f"Filepath: {row['Filepath']}")
    
    filepath = str(row['Filepath'])
    if os.path.exists(filepath):
        # Display the image directly in the notebook output
        display(Image(filename=filepath, width=600))
    else:
        print(f"Image not found at path: {filepath}")

In [ ]:
# ==========================================
# CELL 2: BATCH REPLACE & SAVE
# ==========================================

# 1. Define the pair you want to fix
TARGET_CATEGORY = "Insert Category Here"
TARGET_SUBCAT = "Insert Subcategory Here"

# 2. Define the CORRECT codes they should be mapped to
CORRECT_ROOT_CODE = "NON"
CORRECT_SUB_CODE = "OTH"

# 3. Create the filter condition
condition = (df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)

# 4. Use .loc to safely update the values in the original dataframe
df.loc[condition, 'root_code'] = CORRECT_ROOT_CODE
df.loc[condition, 'sub_code'] = CORRECT_SUB_CODE

print(f"Successfully updated {condition.sum()} rows to {CORRECT_ROOT_CODE} -> {CORRECT_SUB_CODE}.")

# 5. Export the cleaned data to CSV
output_filename = 'label_cleaned.csv'
df.to_csv(output_filename, index=False)
print(f"Cleaned dataset exported as: {output_filename}")

## Evaluation

In [ ]:
import math
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from openai import AzureOpenAI

# ==========================================
# 1. DEFINE STRUCTURED OUTPUT SCHEMAS
# ==========================================

class TriageOutput(BaseModel):
    chain_of_thought: str
    routing_decision: Literal["medical", "processable_non_medical", "trash_others"]

class NonMedSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "financial_bankstatement", "financial_bookbank", 
        "financial_companyregistration", "financial_others",
        "id_driverlicense", "id_fatca_w9", "id_foreignerconfirmationform",
        "id_foreigner_nationalid", "id_fullthaivisa", "id_passport",
        "id_passportstamp", "id_statelessid", "id_thainationalid", 
        "id_workpermit", "id_others"
    ]

class MedSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "medical_clinical", "medical_healthcheck", 
        "medical_lab", "medical_others"
    ]

# ==========================================
# 2. DEFINE PROMPT VARIABLES
# ==========================================

# --- AGENT 1: TRIAGE PROMPTS ---
AGENT1_SYS_PROMPT = """You are a Triage Routing Agent. Your sole job is to categorize incoming messy OCR text into one of three macro-buckets.
Focus heavily on structural elements, broad document headers, and high-level vocabulary context. Do not search for or process specific personal identifiers (PII).

You must route to exactly one of these choices:
1. 'medical': If the document contains clinical notes, prescriptions, lab metrics, health check parameters, hospital names, or physician headers.
2. 'processable_non_medical': If the document appears to be an official ID, passport, visa, work permit, tax form, bank statement, bookbank page, or corporate registration.
3. 'trash_others': If the document is an e-form insurance application, a simple portrait photo placeholder, an unrelated document, or contains completely unparseable structure that does not fit business categories.

Provide your brief macro-level reasoning in 'chain_of_thought' before selecting your choice."""

AGENT1_USER_PROMPT = "Evaluate and triage the following raw OCR text:\n{ocr_text}"


# --- AGENT 2: NON-MED SPECIALIST PROMPTS ---
AGENT2_SYS_PROMPT = """You are an expert Non-Medical Document Specialist. You receive text that has already been verified as a valid Financial or Identification document. Your task is to perform granular classification.

Analyze the layout clues, boilerplate language, and structural anchors. Do not look for or process private personal data. 

### Subclass Evaluation Guides:
- financial_bankstatement: Look for running transactional ledgers, money transfers, account balances, and mobile banking app markers.
- financial_bookbank: Look for static account identity headers (bilingual accounts, branches, bank names) without long subsequent lists of transfer ledgers.
- financial_companyregistration: Look for formal commercial corporate registry verbiage and certification signatures.
- id_fatca_w9: Look for specific US tax withholding terminology, backup withholding rules, and entity declaration sections.
- id_passport / id_fullthaivisa / id_passportstamp: Look for country codes, immigration control stamps, arrival/departure indicators, or explicit visa categories.
- id_workpermit: Look for labor authorization rules and official employment permission contexts.
- id_thainationalid / id_foreigner_nationalid: Look for national demographic card headers and identity issuing text blocks.

Provide your step-by-step structural rationale in 'chain_of_thought' before selecting the final leaf node subcategory."""

AGENT2_USER_PROMPT = "Perform deep classification on this verified processable text:\n{ocr_text}"


# --- AGENT 3: MEDICAL SPECIALIST PROMPTS ---
AGENT3_SYS_PROMPT = """You are an expert Medical Document Specialist. You receive text that has already been verified as an official medical or healthcare document. Your task is to perform granular classification.

Analyze clinical boilerplate language, diagnostic formats, and structural indicators. Do not attempt to analyze or extract confidential patient names or raw medical history.

### Subclass Evaluation Guides:
- medical_clinical: Look for outpatient/inpatient notes (OPD/IPD), physical exam details, hospital visit dates, physician signatures, and medication dosages/frequencies (e.g., "mg", "ครั้ง").
- medical_healthcheck: Look for standardized checkup checklists, physiological summary metrics (e.g., "min", "bpm"), and categorical evaluations of bodily systems (e.g., "normal", "abnormal", "ปกติ", "ไม่พบ").
- medical_lab: Look for quantitative lab panels, chemical compound analysis, test execution timestamps, measurement metrics (e.g., "mg/dL"), or diagnostic presence indicators (e.g., "positive", "negative").
- medical_others: Select this if the document is clearly medical but does not visually or structurally conform to an OPD/IPD sheet, a regular checkup report, or a quantitative lab metric page.

Provide your step-by-step structural rationale in 'chain_of_thought' before selecting the final leaf node subcategory."""

AGENT3_USER_PROMPT = "Perform deep clinical classification on this verified medical text:\n{ocr_text}"


# ==========================================
# 3. CORE COGNITIVE PIPELINE
# ==========================================

def calculate_confidence(response_logprobs) -> float:
    """Extracts and averages log probabilities to return a mathematical percentage."""
    if not response_logprobs or not response_logprobs.content:
        return 0.0
    token_logprobs = [t.logprob for t in response_logprobs.content if t.logprob is not None]
    if not token_logprobs:
        return 0.0
    avg_logprob = sum(token_logprobs) / len(token_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def run_cascade_pipeline(ocr_text: str, client: AzureOpenAI, model_name: str = "gpt-4o") -> dict:
    """Executes the three-stage cascading architecture on a single piece of text."""
    
    # ----------------------------------------------------
    # STAGE 1: Triage Routing
    # ----------------------------------------------------
    triage_response = client.beta.chat.completions.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": AGENT1_SYS_PROMPT},
            {"role": "user", "content": AGENT1_USER_PROMPT.format(ocr_text=ocr_text)}
        ],
        response_format=TriageOutput,
        logprobs=True,
        top_logprobs=1
    )
    
    triage_data = triage_response.choices[0].message.parsed
    triage_conf = calculate_confidence(triage_response.choices[0].logprobs)
    route = triage_data.routing_decision
    
    # ----------------------------------------------------
    # STAGE 2A: Conditional Non-Med Processing
    # ----------------------------------------------------
    if route == "processable_non_medical":
        spec_response = client.beta.chat.completions.parse(
            model=model_name,
            messages=[
                {"role": "system", "content": AGENT2_SYS_PROMPT},
                {"role": "user", "content": AGENT2_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=NonMedSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_confidence(spec_response.choices[0].logprobs)
        
        return {
            "triage_decision": route,
            "triage_reason": triage_data.chain_of_thought,
            "triage_confidence": triage_conf,
            "final_subcategory": spec_data.subcategory,
            "specialist_reason": spec_data.chain_of_thought,
            "specialist_confidence": spec_conf
        }
    
    # ----------------------------------------------------
    # STAGE 2B: Conditional Med Processing
    # ----------------------------------------------------
    elif route == "medical":
        spec_response = client.beta.chat.completions.parse(
            model=model_name,
            messages=[
                {"role": "system", "content": AGENT3_SYS_PROMPT},
                {"role": "user", "content": AGENT3_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=MedSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_confidence(spec_response.choices[0].logprobs)
        
        return {
            "triage_decision": route,
            "triage_reason": triage_data.chain_of_thought,
            "triage_confidence": triage_conf,
            "final_subcategory": spec_data.subcategory,
            "specialist_reason": spec_data.chain_of_thought,
            "specialist_confidence": spec_conf
        }
    
    # If routed to Trash, exit early without calling any specialist agent
    return {
        "triage_decision": route,
        "triage_reason": triage_data.chain_of_thought,
        "triage_confidence": triage_conf,
        "final_subcategory": f"routed_to_{route}",
        "specialist_reason": "Skipped - Cleaned out as unstructured noise by triage agent.",
        "specialist_confidence": 100.0
    }

# ==========================================
# 4. DATAFRAME EVALUATION EXECUTION
# ==========================================

def evaluate_experiment(csv_path: str, endpoint: str, api_key: str):
    """Reads the evaluation data, executes the cascade pipeline, and returns results."""
    
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2024-08-01-preview"
    )
    
    df = pd.read_csv(csv_path)
    experimental_results = []
    
    print(f"Starting complete cascade experiment on {len(df)} documents...")
    
    for idx, row in df.iterrows():
        ocr_text = str(row['ocr_text'])
        
        # Run entire multi-agent cascade pipeline
        res = run_cascade_pipeline(ocr_text, client)
        
        # Merge metrics back with the original ground-truth markers
        combined_row = {
            "input_category_label": row.get('category'),
            "input_subcategory_label": row.get('subcategory'),
            **res
        }
        experimental_results.append(combined_row)
        
        print(f"[{idx+1}/{len(df)}] Triage Macro: {res['triage_decision']} -> Final Leaf Subclass: {res['final_subcategory']}")
        
    output_df = pd.DataFrame(experimental_results)
    output_df.to_csv("experiment_full_cascade_results.csv", index=False)
    print("\nFull evaluation pipeline complete. Results stored in 'experiment_full_cascade_results.csv'")
    return output_df

# To run in your notebook cell:
# df_results = evaluate_experiment("test_data.csv", "https://your-endpoint.openai.azure.com/", "your-api-key")